# 02.02 — Schema Visualization

The `orthograph.visualization` package renders orthograph data structures for human consumption. It supports two output formats:

| Format | Function suffix | Dependencies | Use case |
|--------|----------------|-------------|----------|
| **Mermaid** | `_to_mermaid` | None | Embeddable in markdown, notebooks, docs |
| **Plain text** | `_to_text` | None | Terminal output, logs, CI artifacts |

Three input types can be visualized:

| Input | Mermaid | Text |
|-------|---------|------|
| `GraphDefinition` (schema) | `model_to_mermaid` / `display_mermaid` | `model_to_text` |
| `GraphProfile` (inspection) | -- | `profile_to_text` |
| `ValidationResult` (validation) | -- | `result_to_text` |

This notebook demonstrates every renderer using the filmography domain.

In [1]:
import networkx as nx

from orthograph.api.visualization import render_model, render_profile, render_result
from orthograph.backends.networkx.inspector import NetworkxInspector
from orthograph.comparison.engine import compare_profile_to_definition
from orthograph.visualization.mermaid import display_mermaid, model_to_mermaid
from orthograph.visualization.text import model_to_text, profile_to_text, result_to_text

## Setup: model, graph, profile, and validation result

We define the filmography model, build a NetworkX graph with intentionally incomplete data, inspect it, and validate it. This gives us all three input types for the renderers.

In [2]:
from shared.filmography import FILMOGRAPHY_MODEL


graph_definition = FILMOGRAPHY_MODEL
print("Model:", graph_definition.name, "| nodes:", sorted(graph_definition.node_labels))

Model: Filmography | nodes: ['City', 'Movie', 'Person']


In [3]:
# Build a graph with intentionally incomplete data
G = nx.MultiDiGraph()

G.add_node("p1", __label__="Person", name="Alice", age=30, email="alice@example.com")
G.add_node("p2", __label__="Person", name="Bob", age=45)
G.add_node("p3", __label__="Person", name="Charlie")  # missing 'age'

G.add_node("m1", __label__="Movie", title="The Matrix", year=1999, rating=8.7)
G.add_node("m2", __label__="Movie", title="Inception", year=2010)

G.add_node("c1", __label__="City", name="Los Angeles", country="USA")

G.add_edge("p1", "m1", __label__="ACTED_IN", role="Trinity")
G.add_edge("p2", "m1", __label__="ACTED_IN", role="Morpheus")
G.add_edge("p1", "m2", __label__="ACTED_IN", role="Ariadne")
G.add_edge("p2", "m2", __label__="DIRECTED")
G.add_edge("p1", "c1", __label__="LIVES_IN")
G.add_edge("p2", "c1", __label__="LIVES_IN")

# Inspect and validate
profile = NetworkxInspector().inspect(G)
result = compare_profile_to_definition(profile, graph_definition)

print(
    f"Model:    {graph_definition.name} ({len(graph_definition.node_types)} node types, {len(graph_definition.relationship_types)} relationship types)"
)
print(
    f"Profile:  {sum(ntp.count for ntp in profile.node_type_profiles.values())} nodes, {sum(rtp.count for rtp in profile.rel_type_profiles.values())} relationships"
)
print(
    f"Result:   {'PASS' if result.is_valid else 'FAIL'} ({len(result.errors)} errors, {len(result.warnings)} warnings)"
)

Model:    Filmography (3 node types, 3 relationship types)
Profile:  6 nodes, 6 relationships
Result:   PASS (0 errors, 0 warnings)


## Model visualization: Mermaid

`model_to_mermaid` renders the schema definition as a Mermaid diagram. Node boxes show properties with type, required/optional markers, and UID highlighting. Edges show the relationship type, any properties, and source/target cardinality.

In [4]:
print(model_to_mermaid(graph_definition))

graph TD
    Person["Person<br>name: str UID, born: int?"]
    Movie["Movie<br>title: str UID, released: int?, year: int?"]
    City["City<br>name: str UID"]
    Person -->|ACTED_IN<br>role: str<br>0..* : 0..*| Movie
    Person -->|DIRECTED<br>0..* : 0..*| Movie
    Person -->|LIVES_IN<br>0..* : 0..*| City


## Model visualization: plain text

`model_to_text` renders the same information as a structured text table. Useful for terminal output and logs where Mermaid rendering is not available.

In [5]:
print(model_to_text(graph_definition))

Model: Filmography

Node Types
------------------------------------------------------------
  Person (optional)
    name: str (required) [UID]
    born: int (optional)

  Movie (optional)
    title: str (required) [UID]
    released: int (optional)
    year: int (optional)

  City (optional)
    name: str (required) [UID]

Relationship Types
------------------------------------------------------------
  ACTED_IN: Person --> Movie
    cardinality: [0..*] source, [0..*] target
    role: str (required)

  DIRECTED: Person --> Movie
    cardinality: [0..*] source, [0..*] target

  LIVES_IN: Person --> City
    cardinality: [0..*] source, [0..*] target



## Profile visualization: plain text

`profile_to_text` renders a `GraphProfile` as a text table showing instance counts, property completeness percentages, mandatory/partial flags, observed types, and cardinality statistics.

In [6]:
print(profile_to_text(profile))

Profile: networkx
Timestamp: 2026-06-19 00:24:25.411877

Node Types
------------------------------------------------------------
  City (1 instances)
    country: 100% complete (1/1) [mandatory] types=[str]
    name: 100% complete (1/1) [mandatory] types=[str]

  Movie (2 instances)
    rating: 50% complete (1/2) [partial] types=[float]
    title: 100% complete (2/2) [mandatory] types=[str]
    year: 100% complete (2/2) [mandatory] types=[int]

  Person (3 instances)
    age: 67% complete (2/3) [partial] types=[int]
    email: 33% complete (1/3) [partial] types=[str]
    name: 100% complete (3/3) [mandatory] types=[str]

Relationship Types
------------------------------------------------------------
  ACTED_IN (3 instances)
    sources: ['Person']
    targets: ['Movie']
    cardinality: min=1, max=2, avg=1.5, sample_size=2
    role: 100% complete (3/3) types=[str]

  DIRECTED (1 instances)
    sources: ['Person']
    targets: ['Movie']
    cardinality: min=1, max=1, avg=1.0, sample_siz

## Validation result visualization: plain text

`result_to_text` renders a `ValidationResult` as a severity-coded summary. Issues are grouped by entity, with `[ERROR]`, `[WARNING]`, and `[INFO]` prefixes.

In [7]:
print(result_to_text(result))

Validation: PASS
  Errors: 0, Warnings: 0, Total issues: 4

Issues
------------------------------------------------------------
  node:City.country
    [INFO] UNEXPECTED_PROPERTY: Property 'country' on City found in profile but not in model

  node:Movie.rating
    [INFO] UNEXPECTED_PROPERTY: Property 'rating' on Movie found in profile but not in model

  node:Person.age
    [INFO] UNEXPECTED_PROPERTY: Property 'age' on Person found in profile but not in model

  node:Person.email
    [INFO] UNEXPECTED_PROPERTY: Property 'email' on Person found in profile but not in model



## The explicit render functions

`render_model`, `render_profile`, and `render_result` are the explicit per-object entry points. Each accepts a keyword-only `format=` argument (``RenderFormat.TEXT`` or ``RenderFormat.MERMAID`` where supported).

In [8]:
# Model as mermaid
output = render_model(graph_definition, format="mermaid")
print(f"render_model(graph_definition, format='mermaid') -> {len(output)} chars")
print()

# Profile as text
output = render_profile(profile)
print(f"render_profile(profile) -> {len(output)} chars")
print()

# Result as text
output = render_result(result)
print(f"render_result(result) -> {len(output)} chars")

render_model(graph_definition, format='mermaid') -> 307 chars

render_profile(profile) -> 1130 chars

render_result(result) -> 579 chars


## Inline Mermaid rendering with `display_mermaid`

`display_mermaid` renders a Mermaid diagram as an inline image in a Jupyter notebook. It uses the [mermaid.ink](https://mermaid.ink) service to convert the diagram text to a PNG image (requires an internet connection).

It accepts a raw Mermaid string, a `GraphDefinition`, or a `GraphProfile` -- the conversion to Mermaid text is handled automatically.

In [9]:
# Render the model schema as an inline diagram
display_mermaid(graph_definition)

In [10]:
# Also works with raw Mermaid strings
display_mermaid("graph LR\n    A --> B --> C")

## Side-by-side: schema vs observed

A useful pattern is to render the model schema and the observed profile side by side, so you can compare what the schema *expects* with what the data *contains*.

In [11]:
print("=" * 60)
print("SCHEMA (model definition)")
print("=" * 60)
print(model_to_text(graph_definition))

print("=" * 60)
print("OBSERVED (graph profile)")
print("=" * 60)
print(profile_to_text(profile))

print("=" * 60)
print("VALIDATION")
print("=" * 60)
print(result_to_text(result))

SCHEMA (model definition)
Model: Filmography

Node Types
------------------------------------------------------------
  Person (optional)
    name: str (required) [UID]
    born: int (optional)

  Movie (optional)
    title: str (required) [UID]
    released: int (optional)
    year: int (optional)

  City (optional)
    name: str (required) [UID]

Relationship Types
------------------------------------------------------------
  ACTED_IN: Person --> Movie
    cardinality: [0..*] source, [0..*] target
    role: str (required)

  DIRECTED: Person --> Movie
    cardinality: [0..*] source, [0..*] target

  LIVES_IN: Person --> City
    cardinality: [0..*] source, [0..*] target

OBSERVED (graph profile)
Profile: networkx
Timestamp: 2026-06-19 00:24:25.411877

Node Types
------------------------------------------------------------
  City (1 instances)
    country: 100% complete (1/1) [mandatory] types=[str]
    name: 100% complete (1/1) [mandatory] types=[str]

  Movie (2 instances)
    rati